# Behaviour Under Uniaxial Loading — The Tensile Test

*Plasticity — Block I, Section 1. Companion interactive notebook.*

**Plastic behaviour** is characterised by:
* **permanent** (irreversible) deformation, and
* deformation that **depends on the loading history** $\;\Rightarrow\;$ it is described by an
  *incremental* theory.

Throughout Block I we assume an **ideal plastic solid**:
* isotropy,
* no elastic hysteresis,
* **no Bauschinger effect** (same yield stress in tension and compression),
* time-independence.

The **simple tensile test** is the fundamental experiment used to characterise a material
and to calibrate every plasticity model that follows. This notebook builds it up
interactively:

1. **§1.1** engineering stress & strain,
2. **§1.2** true stress & strain,
3. **§1.3** tensile instability & necking (Considère),
4. **§1.4** empirical stress–strain laws.

The sliders update the figure **in real time**. Run every cell top to bottom
(**Kernel ▸ Restart Kernel and Run All Cells**). Each figure has a toolbar (magnifier to
**zoom**, hand to pan, home to reset the view).

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider
from IPython.display import display

# --- LaTeX-style typography (Computer Modern) for every plot ---
# Computer Modern is LaTeX's default font, so the figures match a LaTeX document
# with no TeX installation required (works on Binder as-is).
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['cmr10', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif',
    'axes.formatter.use_mathtext': True,
    'axes.unicode_minus': False,
    'figure.dpi': 110,
    'font.size': 12,
})

OFFSET = 0.002   # 0.2% conventional plastic-strain offset for the yield stress


def panel(update, sliders, fig):
    # Shows the sliders above a PERSISTENT figure.
    # The figure is created once; moving a slider only redraws its contents
    # (the canvas is not rebuilt), so the response is immediate.
    fig.canvas.header_visible = False
    out = widgets.interactive_output(update, sliders)
    display(widgets.VBox(list(sliders.values())), fig.canvas, out)

## 1.1 The simple tensile test

A specimen of initial length $L_0$ and initial cross-section $A_0$ is pulled by a force
$F$, elongating by $\Delta L$.

**Engineering stress** (referred to the *initial* area):

$$S=\frac{F}{A_0}\qquad\left[\tfrac{\mathrm N}{\mathrm m^2}=\mathrm{Pa}\right]$$

**Engineering strain** (dimensionless):

$$e=\frac{\Delta L}{L_0}$$

The $S\!-\!e$ curve shows an **elastic** region (slope $=E$), a **plastic** region, and
finally **necking** (*estricción*) up to fracture. The material properties read from it are:

* the **elastic modulus** $E$,
* the **tensile strength** $S_\max$ (the peak of the engineering curve, UTS),
* the **conventional yield stress** $S_Y=F_x/A_0$, defined at a plastic-strain offset
  $x$ (customarily $x=0.2\%=0.002$).

## 1.2 True stress and true strain

Unlike engineering stress (referred to $A_0$), the **true stress** is referred to the
*instantaneous* area $A$:

$$\sigma=\frac{F}{A}.$$

Engineering strain is **not additive**. The **true strain** is defined incrementally and
*is* additive:

$$\mathrm d\varepsilon=\frac{\mathrm dL}{L}\;\Rightarrow\;
\varepsilon=\int_{L_0}^{L}\frac{\mathrm dL}{L}=\ln\frac{L}{L_0}.$$

Assuming **plastic incompressibility** ($V=A_0L_0=AL=\text{const}$) gives the conversion
between true and engineering magnitudes:

$$\boxed{\;\varepsilon=\ln(1+e)\qquad\sigma=S\,(1+e)\;}$$

These hold while deformation is **uniform** (up to necking). The cell below plots both
curves from the same material and shows how they diverge as strain grows.

In [ ]:
def flow_curve(E, Y, n, eps):
    """Elastic then power-law (Hollomon-type) hardening, true stress vs true strain.
    Continuous at yield: sigma(eps_y)=Y, with K = Y / eps_y**n."""
    eps_y = Y/E
    K = Y/eps_y**n
    sig = np.where(eps <= eps_y, E*eps, K*eps**n)
    return sig, eps_y, K


# --- PERSISTENT figure (created once) ---
plt.ioff()
fig1, ax1 = plt.subplots(figsize=(8.2, 5.6))
plt.ion()


def plot_eng_vs_true(E_GPa=200.0, Y=250.0, n=0.15, eps_max=0.45):
    ax = ax1
    ax.clear()                                         # only the contents are redrawn

    E = E_GPa*1.0e3
    eps = np.linspace(1e-6, eps_max, 400)
    sig, eps_y, K = flow_curve(E, Y, n, eps)          # TRUE stress vs TRUE strain

    e = np.expm1(eps)                                  # engineering strain  e = exp(eps)-1
    S = sig*np.exp(-eps)                               # engineering stress  S = sigma/(1+e)

    eps_neck = n                                       # Considere: necking at eps = n
    sig_neck = K*eps_neck**n
    S_max = sig_neck*np.exp(-eps_neck)                 # UTS (peak engineering stress)
    e_neck = np.expm1(eps_neck)

    ax.plot(eps, sig, color='crimson', lw=2.2, label=r'true  $\sigma\,(\varepsilon)$')

    m = e <= e_neck                                    # uniform (valid) part
    ax.plot(e[m],  S[m],  color='steelblue', lw=2.2, label=r'engineering  $S\,(e)$')
    ax.plot(e[~m], S[~m], color='steelblue', lw=1.6, ls='--', alpha=0.5,
            label=r'eng. (post-necking, non-uniform)')

    # 0.2% offset construction for the yield stress
    off = E*(eps - OFFSET)
    ax.plot(eps[off <= Y*1.05], off[off <= Y*1.05], color='0.6', lw=1.0, ls=':')
    ax.plot(eps_y, Y, 'o', color='k', ms=6)
    ax.annotate(r'$S_Y=%.0f$' % Y, (eps_y, Y), textcoords='offset points',
                xytext=(8, -12), fontsize=11)

    # UTS / necking markers
    ax.plot(e_neck, S_max, 's', color='steelblue', ms=9, zorder=5)
    ax.annotate(r'$S_{\max}=%.0f$' % S_max, (e_neck, S_max),
                textcoords='offset points', xytext=(8, 6), fontsize=11, color='steelblue')
    ax.plot(eps_neck, sig_neck, 'o', color='crimson', ms=8, zorder=5)
    ax.axvline(eps_neck, color='0.7', lw=0.8, ls=':')
    ax.annotate(r'necking  $\varepsilon=n=%.2f$' % n, (eps_neck, 0),
                textcoords='offset points', xytext=(6, 8), fontsize=10, color='0.35')

    ax.set_xlim(0, eps_max)
    ax.set_ylim(0, max(sig.max(), S_max)*1.1)
    ax.set_xlabel(r'strain  $\varepsilon,\;e$  (-)')
    ax.set_ylabel(r'stress  $\sigma,\;S$  (MPa)')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right', fontsize=10)
    ax.set_title(r'Engineering vs. true curve:  $\sigma=S(1+e)$,  $\varepsilon=\ln(1+e)$',
                 fontsize=12)
    fig1.canvas.draw_idle()                            # immediate refresh


panel(plot_eng_vs_true, dict(
    E_GPa=FloatSlider(value=200, min=50, max=400, step=10,
                      description='E (GPa)', style={'description_width':'initial'}),
    Y=FloatSlider(value=250, min=50, max=600, step=10,
                  description='yield  Y (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='hardening exp.  n', style={'description_width':'initial'}),
    eps_max=FloatSlider(value=0.45, min=0.1, max=0.8, step=0.05,
                        description='max strain', style={'description_width':'initial'}),
), fig1)

## 1.3 Tensile instability and necking

The load is $F=\sigma A$. During the test $\sigma$ **increases** (hardening) while $A$
**decreases** (incompressibility). The load reaches a **maximum** — beyond it deformation
localises into a **neck** and the engineering stress falls. Setting $\mathrm dF=0$:

$$\mathrm dF=\mathrm d(\sigma A)=0
\;\Rightarrow\;\frac{\mathrm d\sigma}{\sigma}=-\frac{\mathrm dA}{A}=\mathrm d\varepsilon
\;\Rightarrow\;\boxed{\dfrac{\mathrm d\sigma}{\mathrm d\varepsilon}=\sigma}\quad\text{(Considère)}$$

For a Hollomon law $\sigma=K\varepsilon^{\,n}$ this gives the **uniform elongation**

$$nK\varepsilon^{\,n-1}=K\varepsilon^{\,n}\;\Rightarrow\;\varepsilon_{\text{neck}}=n,$$

i.e. the true strain at necking equals the strain-hardening exponent. Necking is a
**structural** phenomenon (it depends on geometry *and* material), not a pure material
property.

The **Considère construction**: at the necking point the tangent to $\sigma(\varepsilon)$
has a *subtangent equal to 1*, so the tangent line drawn at that point crosses
$\sigma=0$ at $\varepsilon=n-1$.

In [ ]:
# --- PERSISTENT figure (two panels) ---
plt.ioff()
fig2, (ax2L, ax2R) = plt.subplots(1, 2, figsize=(12.0, 5.0), constrained_layout=True)
plt.ion()


def plot_considere(K=520.0, n=0.15, eps_max=0.5):
    axL, axR = ax2L, ax2R
    axL.clear(); axR.clear()

    eps = np.linspace(1e-4, eps_max, 400)
    sig = K*eps**n                       # Hollomon (plastic flow) true curve

    eps_neck = n
    sig_neck = K*eps_neck**n
    S = sig*np.exp(-eps)                 # engineering stress
    e = np.expm1(eps)
    S_max = sig_neck*np.exp(-eps_neck)
    e_neck = np.expm1(eps_neck)

    # --- LEFT: Considere tangent construction on the true curve ---
    axL.plot(eps, sig, color='crimson', lw=2.2, label=r'$\sigma=K\varepsilon^{\,n}$')
    xt = np.array([eps_neck - 1.0, min(eps_max, eps_neck + 0.15)])
    axL.plot(xt, sig_neck*(1.0 + (xt - eps_neck)), color='0.4', lw=1.3, ls='--',
             label='tangent (subtangent $=1$)')
    axL.plot(eps_neck, sig_neck, 'o', color='crimson', ms=9, zorder=5)
    axL.plot(eps_neck - 1.0, 0.0, 'o', color='0.4', ms=5)
    axL.axvline(eps_neck, color='0.7', lw=0.8, ls=':')
    axL.set_xlim(eps_neck - 1.05, eps_max)
    axL.set_ylim(0, sig.max()*1.1)
    axL.set_xlabel(r'true strain  $\varepsilon$  (-)')
    axL.set_ylabel(r'true stress  $\sigma$  (MPa)')
    axL.grid(alpha=0.3); axL.legend(loc='upper left', fontsize=10)
    axL.set_title(r'Considere construction:  $\varepsilon_{\rm neck}=n=%.2f$' % n, fontsize=12)

    # --- RIGHT: the load (engineering stress) has a maximum at necking ---
    axR.plot(e, S, color='steelblue', lw=2.2, label=r'engineering  $S\,(e)$')
    axR.plot(e_neck, S_max, 's', color='steelblue', ms=10, zorder=5)
    axR.annotate(r'$S_{\max}$ (UTS)', (e_neck, S_max), textcoords='offset points',
                 xytext=(8, -4), fontsize=11, color='steelblue')
    axR.axvline(e_neck, color='0.7', lw=0.8, ls=':')
    axR.set_xlim(0, np.expm1(eps_max))
    axR.set_ylim(0, S.max()*1.2)
    axR.set_xlabel(r'engineering strain  $e$  (-)')
    axR.set_ylabel(r'engineering stress  $S$  (MPa)')
    axR.grid(alpha=0.3); axR.legend(loc='lower right', fontsize=10)
    axR.set_title(r'Maximum load $\;\Rightarrow\;$ onset of necking', fontsize=12)

    fig2.canvas.draw_idle()


panel(plot_considere, dict(
    K=FloatSlider(value=520, min=200, max=1200, step=20,
                  description='strength  K (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='hardening exp.  n', style={'description_width':'initial'}),
    eps_max=FloatSlider(value=0.5, min=0.2, max=0.9, step=0.05,
                        description='max strain', style={'description_width':'initial'}),
), fig2)

## 1.4 Empirical stress–strain laws

To represent the plastic part of the $\sigma\!-\!\varepsilon$ curve analytically, several
empirical laws are used:

* **Hollomon (power law):** $\;\sigma=K\,\varepsilon_p^{\,n}\;$ — $K$ is the strength
  coefficient and $n\in[0,1]$ the strain-hardening exponent ($n=0$ is perfectly plastic).
* **Ludwik:** $\;\sigma=\sigma_Y+K\,\varepsilon_p^{\,n}\;$ — Hollomon with an additive yield
  term, so the plastic branch **starts at the yield stress** ($\sigma=\sigma_Y$ at
  $\varepsilon_p=0$) rather than at zero.
* **Ramberg–Osgood:** a smooth elastic–plastic transition with no sharp yield point.
  A common form is $\;\varepsilon=\dfrac{\sigma}{E}+\alpha\!\left(\dfrac{\sigma}{Y}\right)^{m}$
  (the offset $\alpha=0.002$ passes through the $0.2\%$ yield).
* **Idealisations:** *elastic–perfectly plastic* and *elastic–linear hardening*, used as
  simple models depending on the problem.

The cell below overlays all of them for the same $E$ and $Y$, so you can see how $n$, the
hardening modulus $H$, and the Ramberg–Osgood exponent $m$ change the shape.

In [ ]:
# --- PERSISTENT figure ---
plt.ioff()
fig3, ax3 = plt.subplots(figsize=(8.2, 5.6))
plt.ion()


def plot_laws(E_GPa=200.0, Y=250.0, n=0.15, H=1500.0, K_L=500.0, m=8.0, eps_max=0.05):
    ax = ax3
    ax.clear()

    E = E_GPa*1.0e3
    eps_y = Y/E
    eps = np.linspace(1e-6, eps_max, 400)
    epsp = np.clip(eps - eps_y, 0.0, None)   # plastic strain (approx.)

    # elastic - perfectly plastic
    epp = np.where(eps <= eps_y, E*eps, Y)
    # elastic - linear hardening
    lin = np.where(eps <= eps_y, E*eps, Y + H*(eps - eps_y))
    # Hollomon (continuous at yield): sigma = K eps^n
    K = Y/eps_y**n
    hol = np.where(eps <= eps_y, E*eps, K*eps**n)
    # Ludwik: sigma = Y + K_L * eps_p^n  (starts at Y when eps_p = 0)
    lud = np.where(eps <= eps_y, E*eps, Y + K_L*epsp**n)

    ax.plot(eps, epp, lw=2.0, label='elastic - perfectly plastic')
    ax.plot(eps, lin, lw=2.0, label=r'elastic - linear hardening ($H$)')
    ax.plot(eps, hol, lw=2.0, label=r'Hollomon  $\sigma=K\varepsilon^{\,n}$')
    ax.plot(eps, lud, lw=2.0, label=r'Ludwik  $\sigma=\sigma_Y+K\varepsilon_p^{\,n}$')

    # Ramberg-Osgood: eps = sigma/E + 0.002 (sigma/Y)^m  -> build sigma-axis then plot
    sig_ro = np.linspace(1e-6, max(Y*1.8, hol.max()), 300)
    eps_ro = sig_ro/E + OFFSET*(sig_ro/Y)**m
    ro_mask = eps_ro <= eps_max
    ax.plot(eps_ro[ro_mask], sig_ro[ro_mask], lw=2.0, ls='--',
            label=r'Ramberg-Osgood  ($m$)')

    ax.axhline(Y, color='0.7', lw=0.8, ls=':')
    ax.plot(eps_y, Y, 'ko', ms=5)
    ax.annotate(r'$Y=%.0f$' % Y, (eps_y, Y), textcoords='offset points',
                xytext=(8, -14), fontsize=11)

    ax.set_xlim(0, eps_max)
    ax.set_ylim(0, max(lin.max(), hol.max(), lud.max(), Y*1.6)*1.05)
    ax.set_xlabel(r'strain  $\varepsilon$  (-)')
    ax.set_ylabel(r'stress  $\sigma$  (MPa)')
    ax.grid(alpha=0.3); ax.legend(loc='lower right', fontsize=10)
    ax.set_title('Empirical stress-strain laws (same $E$, $Y$)', fontsize=12)
    fig3.canvas.draw_idle()


panel(plot_laws, dict(
    E_GPa=FloatSlider(value=200, min=50, max=400, step=10,
                      description='E (GPa)', style={'description_width':'initial'}),
    Y=FloatSlider(value=250, min=50, max=600, step=10,
                  description='yield  Y (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='Hollomon  n', style={'description_width':'initial'}),
    H=FloatSlider(value=1500, min=0, max=8000, step=100,
                  description='hardening  H (MPa)', style={'description_width':'initial'}),
    K_L=FloatSlider(value=500, min=100, max=2000, step=50,
                    description='Ludwik  K (MPa)', style={'description_width':'initial'}),
    m=FloatSlider(value=8.0, min=2.0, max=25.0, step=1.0,
                  description='R-O exponent  m', style={'description_width':'initial'}),
    eps_max=FloatSlider(value=0.05, min=0.01, max=0.2, step=0.01,
                        description='max strain', style={'description_width':'initial'}),
), fig3)

## Summary — key relations

| Quantity | Engineering | True |
|---|---|---|
| stress | $S=F/A_0$ | $\sigma=F/A=S(1+e)$ |
| strain | $e=\Delta L/L_0$ | $\varepsilon=\ln(1+e)$ |

* Valid while deformation is **uniform** (up to necking); plastic incompressibility assumed.
* **Yield** $S_Y$: from the $0.2\%$ plastic-strain offset.
* **Instability (Considère):** $\dfrac{\mathrm d\sigma}{\mathrm d\varepsilon}=\sigma$;
  for Hollomon $\varepsilon_{\text{neck}}=n$ and $S_{\max}$ is the UTS.
* This uniaxial curve is what calibrates the multiaxial yield criteria (Tresca, von Mises)
  and hardening laws in the next sections.